## Supplement 1: Re-Epoching: Custom Time Windows and Event Selection

This notebook covers how to re-epoch your preprocessed OPM MEG data
when you need a different time window or want to epoch around a specific subset of events.
It picks up from the same ICA-cleaned continuous file (`proc-clean_raw.fif`) used by the other
notebooks.

**What the notebook covers**

1. **Re-epoch with a new time window** — same two conditions, different `tmin`/`tmax`
2. **Re-epoch around a specific event** — restrict to another condition *and* change the window
3. **Average the new epochs** — create an `Evoked` object with `epochs.average()`
4. **Butterfly plot** — visualise the new evoked response

---
### Why re-epoch from the raw file?

MNE `Epochs` objects store data only within the window used when they were created.
There is no way to expand that window after the fact — the samples outside it were
never loaded. Re-epoching from the ICA-cleaned continuous recording lets you choose
any window you like while still benefiting from all the earlier preprocessing
(filtering, HFC denoising, ICA artifact removal).

---
### Which events were epoched for each experiment?
*Attention Oddball*: standard tones, deviant tones / -0.5 to 0.8 window / -0.2 to 0 baseline

*Word Oddball*: standard tones, deviant tones / -0.1 to 1.2 window / -0.1 to 0 baseline

*Piano Tones*: probe / -0.1 to 1.0 window / -0.1 to 0 baseline

*Binding*: fixation, rule, test, test_match, test_non_match, transition, response / -0.1 to 0.4 window / -0.1 to 0 baseline

---
### Running this notebook

```bash
uv run --project $MNE_OPM_DIR jupyter lab
```
Where $MNE_OPM_DIR is the path to your downloaded mne-opm repo.

Use `%matplotlib inline` for static figures (default below).  
Switch to `%matplotlib qt` for interactive plots where you can scroll, zoom, and mark bad epochs.

## 0. Setup

In [ ]:
%matplotlib inline
# %matplotlib qt  # uncomment for interactive scrollable plots

import pathlib
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level('WARNING')  # reduce console noise
print(f'MNE version: {mne.__version__}')

In [ ]:
# ─── USER SETTINGS ───────────────────────────────────────────────────────────
DERIV_DIR = pathlib.Path('/Volumes/NEU502/2026-NEU502B/oddball/data/oddball/bids/derivatives/analysis1__2/sub-001/ses-01/meg')

SUBJECT = 'sub-001'
SESSION = 'ses-01'
TASK    = 'oddball'

# Condition labels — must match what was used during BIDS annotation/epoching
STANDARD_COND = 'standard_onset'
DEVIANT_COND  = 'deviant_onset'
# ─────────────────────────────────────────────────────────────────────────────

# ICA-cleaned continuous file — the starting point for all custom epoching
clean_fif = DERIV_DIR / f'{SUBJECT}_{SESSION}_task-{TASK}_run-01_proc-clean_raw.fif'

print(f'Clean raw : {clean_fif.name}  [exists={clean_fif.exists()}]')

## 1. Load the Cleaned Raw File and Extract Events

We start from the ICA-cleaned continuous recording, which retains all the original
BIDS annotations. `mne.events_from_annotations()` converts those annotations into
the integer event array that `mne.Epochs()` requires.

The event array has shape `(n_events, 3)`:

| Column | Meaning |
|--------|---------|
| 0 | Sample index of the event onset |
| 1 | Previous event ID (legacy; almost always 0 — ignore) |
| 2 | Integer event ID |

`event_id` is a dict mapping human-readable condition names to their integer codes,
e.g. `{'standard_onset': 1, 'deviant_onset': 2}`. You will pass this dict — or a
subset of it — to `mne.Epochs()` to control which events are epoched.

In [ ]:
# Load header only first — saves RAM until we decide on the window
raw_clean = mne.io.read_raw_fif(clean_fif, preload=False)
print(raw_clean)

# Extract events from the BIDS-compatible annotations in the file
events, event_id = mne.events_from_annotations(raw_clean, verbose=False)

print(f'\nTotal events  : {len(events)}')
print(f'Event ID map  : {event_id}')
print()
# Count events per condition
for name, code in event_id.items():
    n = (events[:, 2] == code).sum()
    print(f'  {name:<25s}: {n} events')

## 2. Re-Epoch with a New Time Window

The data was preprocessed with a default window of −500 ms to +800 ms. Suppose you want a tighter window —
for example −200 ms to +500 ms — to reduce epoch length for a decoding analysis,
or a longer window to look at late responses.

Pass the new `tmin` and `tmax` values directly to `mne.Epochs()`. Everything else
follows the same pattern as the pipeline: apply the pre-stimulus baseline, reject
epochs that exceed a peak-to-peak amplitude threshold, and load the data into RAM
with `preload=True`.

> **Tip:** Pass `event_id=event_id` (the full dict) here so that all conditions
> are present in the same `Epochs` object. You can still select individual conditions
> later via `epochs[STANDARD_COND]` or `epochs[DEVIANT_COND]`.

In [ ]:
# ─── New time window ──────────────────────────────────────────────────────────
NEW_TMIN      = -0.200   # seconds before event onset
NEW_TMAX      =  0.500   # seconds after event onset
BASELINE      = (-0.200, 0.0)   # pre-stimulus baseline window (same endpoints as tmin → 0)
REJECT_THRESH = 3e-12    # peak-to-peak rejection threshold in Tesla (3 pT)
# ──────────────────────────────────────────────────────────────────────────────

epochs_new_window = mne.Epochs(
    raw_clean,
    events=events,
    event_id=event_id,       # include all conditions
    tmin=NEW_TMIN,
    tmax=NEW_TMAX,
    baseline=BASELINE,
    picks='mag',             # magnetometer channels only
    reject=dict(mag=REJECT_THRESH),
    preload=True,
    verbose=False,
)

print(epochs_new_window)
print(f'\nTime axis : {epochs_new_window.tmin:.3f} s  →  {epochs_new_window.tmax:.3f} s')
print(f'Samples   : {epochs_new_window.times.shape[0]}')
print()
# Per-condition trial counts after rejection
for cond in event_id:
    n = len(epochs_new_window[cond])
    print(f'  {cond:<25s}: {n} epochs retained')


## 3. Re-Epoch with a New Time Window *and* a Specific Event

Sometimes you only want epochs for one condition — let's look at the blink window — and you want a different window previously used.

Pass a *subset* of `event_id` to `mne.Epochs()` instead of the full dict. Any event
codes not in the dict you provide are simply ignored during epoching.

Below we epoch only the blink events in a wider post-stimulus window (−200 ms to +1500 ms)

In [ ]:
print(event_id)

# ─── New window + single condition ───────────────────────────────────────────
DEV_TMIN = -0.200
DEV_TMAX =  1.500
BLINK_COND = 'blink_trial'   # <-- set this to whatever key appeared in event_id
# ──────────────────────────────────────────────────────────────────────────────

# Restrict event_id to only the deviant condition
blink_trial_id = {BLINK_COND: event_id[BLINK_COND]}
print(f'Epoching around event: {blink_trial_id}')

epochs_blinks = mne.Epochs(
    raw_clean,
    events=events,
    event_id=blink_trial_id,   # <-- only blink events
    tmin=DEV_TMIN,
    tmax=DEV_TMAX,
    baseline=(-0.100, 0.0),
    picks='mag',
    reject=dict(mag=REJECT_THRESH),
    preload=True,
    verbose=False,
)

print(epochs_blinks)
print(f'\nTime axis : {epochs_blinks.tmin:.3f} s  →  {epochs_blinks.tmax:.3f} s')
print(f'Epochs retained: {len(epochs_blinks)}')

### Why does `event_id` filtering happen at epoch creation?

`mne.Epochs()` evaluates the `event_id` dict at construction time: only events whose
integer code appears in the dict values are kept. There is no equivalent of a
post-hoc "filter by event" method — if you want a different set of conditions, create
a new `Epochs` object from the raw file with the appropriate `event_id` subset.

This is different from *condition selection* inside an existing `Epochs` object
(e.g. `epochs[DEVIANT_COND]`), which works on a pre-built object but cannot change
the time window or include events that were excluded during construction.

## 4. Average the New Epochs to Create a New Evoked Object

`epochs.average()` computes the arithmetic mean across all retained trials,
collapsing the `(n_epochs, n_channels, n_times)` array down to `(n_channels, n_times)`.
The result is an `Evoked` object — the same type produced by the pipeline and explored
in notebooks 2 and 3.

We create a new evoked object here:

`evoked_blink` — blink trials from the single-condition `epochs_blinks`

In [ ]:
evoked_blink = epochs_blinks.average(picks='mag')

print('Blink trial (new window):')
print(f'  nave={evoked_blink.nave}  '
      f'tmin={evoked_blink.tmin:.3f} s  tmax={evoked_blink.tmax:.3f} s  '
      f'shape={evoked_blink.data.shape}')

## 5. Butterfly Plot of the New Evoked Object

`evoked.plot()` overlays all channel traces on a shared time axis — a butterfly plot.
We restrict to **Z (radial) channels** here, as in notebook 3.

A few things to notice:

- The time axis now reflects the new window you chose — the plot starts at your
  `tmin` and ends at your `tmax`.
- The `gfp=True` argument adds the **Global Field Power** line (population standard
  deviation across channels), which peaks at moments of spatially distributed activity.


In [ ]:
# Select Z-axis (radial) channels from the evoked object
z_names = [ch for ch in evoked_blink.copy().pick('mag', exclude='bads').ch_names
           if ch.endswith(' Z')]
print(f'Z channels: {len(z_names)}')

evoked_blink_z = evoked_blink.copy().pick_channels(z_names)


In [ ]:
# ── Butterfly plot ───────────────────────────────────────────────────────────
fig = evoked_blink_z.plot(
    picks='mag',
    units=dict(mag='fT'),
    scalings=dict(mag=1e15),
    titles=dict(mag=f'Blink trial — Z channels (n={evoked_blink_z.nave})'),
    gfp=True,
    show=False,
)
for ax in fig.axes:
    ax.axvline(0, color='black', lw=0.8, ls='--', zorder=3, label='blink cue onset')
plt.show()

### To save the new evoked objects for use in other notebooks:

In [ ]:
mne.write_evokeds(
    DERIV_DIR / f'{SUBJECT}_{SESSION}_task-{TASK}_blink_ave.fif',
    [evoked_blink],
    overwrite=True,
)